# Scenario Quickstart

This notebook shows how to use `ScenarioManager` as a standalone workflow for land-cover-change experiments.

It covers:

1. Creating a deforestation or reforestation scenario from a polygon.
2. Visual validation of tree cover, herb cover, and NDVI changes.
3. Recomputing runoff with scenario-specific vegetation inputs.
4. Running streamflow simulation from the scenario runoff outputs.

## Imports

In [ ]:
from pathlib import Path

from bakaano.extensions.scenario import ScenarioManager

## Configure paths

In [ ]:
working_dir = Path("/path/to/working_dir")
study_area = Path("/path/to/study_area.shp")
model_path = working_dir / "models" / "bakaano_model.keras"

working_dir, study_area, model_path

## Initialize ScenarioManager

In [ ]:
sm = ScenarioManager(
    working_dir=working_dir,
    study_area=study_area,
    climate_data_source="ERA5",
)

sm

## Draw a scenario polygon

Create the map, draw a polygon, and keep the map object in memory.

In [ ]:
m = sm.build_draw_map()
m

## Create a scenario from the drawn polygon

In [ ]:
scenario_name = "deforest_50"

metadata = sm.create_land_cover_scenario(
    scenario_name=scenario_name,
    map_obj=m,
    percent_change=50,
    change_type="deforestation",
)

metadata

## Visual validation

First inspect the tree-cover raster change, then compare the NDVI seasonal response through the year.

For NDVI, the scenario uses a threshold-based rule tied to VegET's `0.4` breakpoint:

- `reforestation`: cells inside the polygon with `NDVI <= 0.4` are pushed above `0.4`
- `deforestation`: cells inside the polygon with `NDVI > 0.4` are pushed below `0.4`

The shift size `delta` comes from `percent_change` and is recorded in the scenario metadata.

In [ ]:
sm.plot_scenario_change(scenario_name);

In [ ]:
sm.plot_ndvi_scenario_timeseries(scenario_name);

Inspect the NDVI threshold parameters used for this scenario:

In [ ]:
metadata["ndvi_threshold_rule"]

## Recompute runoff with scenario inputs

This uses the scenario-specific tree cover, herb cover, and NDVI climatology.

In [ ]:
sm.recompute_runoff(
    scenario_name=scenario_name,
    sim_start="2001-01-01",
    sim_end="2010-12-31",
    routing_method="mfd",
)


## Simulate scenario streamflow at points

In [ ]:
result = sm.simulate_streamflow(
    scenario_name=scenario_name,
    model_path=model_path,
    sim_start="2001-01-01",
    sim_end="2010-12-31",
    latlist=[13.8],
    lonlist=[3.0],
    routing_method="mfd",
    area_normalize=True,
)

result

## Outputs

Scenario outputs are written under `working_dir/scenarios/<scenario_name>/`:

- `vcf/mean_tree_cover.tif`
- `vcf/mean_herb_cover.tif`
- `ndvi/daily_ndvi_climatology.pkl`
- `scenario_geometry.geojson`
- `scenario_metadata.json`
- `runoff_output/`
- `predicted_streamflow_data/`